**Data Science and AI in Business**
## *Spring 2026*
Daniel M. Ringel  
www.ringel.ai

## BONUS MATERIALS - **myAI3:** Populating your RAG Database on Pinecone

*January 8, 2026*  
Version 1.1


This is the notebook you will use to upload documents to a Pinecone index, specifically adapted for myAI3.

# Prerequisites

1. First, **install the necessary packages**. We need `unstructured[all-docs]` to parse and chunk documents, `pinecone` for the vector database, `pydantic` for clean code, and `poppler-utils` for presentations.

In [ ]:
!pip install unstructured[all-docs] pinecone pydantic poppler-utils -q


2. **Click on the Key icon and add the secrets** `PINECONE_API_KEY` and `PINECONE_INDEX_HOST`. You can find the index host under `HOST` in the card below when you are looking at your Index.

![image.png](https://ringel.ai/UNC/2026/img/pinecone-index-card.png)

In [ ]:
# Load your API keys from your google userdata
from google.colab import userdata

PINECONE_API_KEY: str = userdata.get("PINECONE_API_KEY")
PINECONE_INDEX_HOST: str = userdata.get("PINECONE_INDEX_HOST")

In [ ]:
# IMPORTANT: Imports for this notebook

from pathlib import Path
from urllib.parse import urlparse, ParseResult
from unstructured.partition.auto import partition, Element
from unstructured.chunking.title import chunk_by_title
from pydantic import BaseModel, Field
from enum import Enum
from uuid import UUID, uuid4
from pinecone import Pinecone
from tqdm.notebook import tqdm

Great, now we have our API keys all set.

# Upload Text/Image/Document

1. Upload the file in the sidebar with the Folder icon
2. When you run the cell below, you will be asked for the file path. The easiest way to get this is to hover over the file you uploaded, click on the 3 dots, then click on "Copy path".
3. Great, now run the cell below!


> Keep in mind that **if you want to upload images**, you must **save them on a webserver** and tell Pinecone what they are and where they are URL!

In [ ]:
### Step 1: Run this cell and you will be asked for inputs

chunk_type: str
while True:
  options: str = ["image", "text"]
  chunk_type: str = input(f"What's the type of the chunk? ({", ".join(options)}) ").strip()
  if chunk_type not in options:
    print("\tInvalid chunk type")
  elif chunk_type == "" or chunk_type is None:
    print("\tYou have to give your chunk a type.")
  else:
    break

file_path: str
while True and chunk_type == "text":
    file_path = input("What's the path to the document? ").strip()
    if Path(file_path).exists():
        print("\tFile found.")
        break
    else:
        print("\tFile not found")

source_name: str
while True:
  source_name: str = input("What's the name of the source? ").strip()
  if source_name == "" or source_name is None:
    print("\tYou have to give your source a name. This name will be needed when deleting the source and when citations are displayed.")
  else:
    break

source_url: str
while True:
  if chunk_type == "image":
    print("WARNING: You are uploading an image, so right click on the image and copy it's ADDRESS (the URL should end with .png, .jpg, etc.)")
  source_url: str = input("What's the URL of the source? ").strip()
  if (source_url == "" or source_url is None) and chunk_type == "text":
    print("\tNo source URL is fine, but users will not be able to click on the citation")
    break
  if (source_url == "" or source_url is None) and chunk_type == "image":
    print("\tIf you are trying to upload an image, you have to upload the URL to the image here.")
  parse_result: ParseResult = urlparse(source_url)
  if parse_result.scheme and parse_result.netloc:
    break
  else:
    print("\tURL is invalid. Give the full address (including http, etc.)")

source_description: str
while True:
  source_description = input("What's the description of the source? ").strip()
  if (source_description == "" or source_description is None) and chunk_type == "text":
    print("\tNo source description is fine for text.")
    break
  if (source_description == "" or source_description is None) and chunk_type == "image":
    print("\tYou need to give a description for images.")
  else:
    break

In [ ]:
# Step 2: ONLY for text, not image descriptions!

# If you are upload an image, there is no need to run this cell

if (chunk_type == "text"):
  print(f"Partitioning '{file_path}'")
  elements: list[Element] = partition(filename=file_path)
  print(f"Found {len(elements)} elements")
  chunks: list[Element] = chunk_by_title(elements=elements)
  print(f"Formed {len(chunks)} chunks from elements")

Now we have chunks, we have to turn them into records. Remember, chunks must be turned into records so we can upsert them into Pinecone.

In [ ]:
# Step 3: Turn chunks into records that are ready to be uploaded to your Pinecone database

class ChunkType(str, Enum):
  IMAGE = "image"
  TEXT = "text"

class Record(BaseModel):
  id: str = Field(default_factory=lambda: str(uuid4()))
  chunk_type: ChunkType
  pre_context: str
  text: str
  post_context: str
  source_name: str
  source_url: str
  source_description: str
  order: int

records: list[dict] = []

if chunk_type == "text":
  for i, chunk in enumerate(chunks):
    pre_context_text = chunks[i-1].text if i > 0 else ""
    post_context_text = chunks[i+1].text if i < len(chunks) - 1 else ""

    record: Record = Record(
        chunk_type=ChunkType.TEXT,
        pre_context=pre_context_text,
        text=chunk.text,
        post_context=post_context_text,
        source_name=source_name,
        source_url=source_url,
        source_description=source_description,
        order=i
    )
    records.append(record.model_dump())
else:
  assert source_description != "" or source_description is not None, "Images need a source description. You did not input one earlier."
  pre_context: str = "The following is a description of an image:"
  post_context: str = "If you want to display the image to the user, use the markdown format for images."
  record: Record = Record(
      chunk_type=ChunkType.IMAGE,
      text=source_description,
      source_name=source_name,
      source_url=source_url,
      source_description=source_description,
      pre_context=pre_context,
      post_context=post_context,
      order=0
  )
  records.append(record.model_dump())
print(records)

Now we have records, we can go ahead and upsert them into Pinecone now.

In [ ]:
# Step 4. Upsert your chunks into Pinecone

pc: Pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(host=PINECONE_INDEX_HOST)

batch_size = 96
print(records)
for i in tqdm(range(0, len(records), batch_size), desc="Upserting records"):
    batch = records[i:i + batch_size]
    index.upsert_records(
        namespace="default",
        records=batch,
    )
print(f"Upserted {len(records)} records in batches to Pinecone.")

# Delete Records by Source Name

If you upserted something into your Pinecone that you need to delete you can run the cell below.

In [ ]:
from pinecone import Pinecone

pc: Pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(host=PINECONE_INDEX_HOST)

NAMESPACE = "default"
METADATA_FIELD = "source_name"

source_name = input(f"What value {METADATA_FIELD} do you want to delete from Pinecone? ").strip()

if not source_name:
    print(f"No {METADATA_FIELD} provided. Aborting.")
else:
    all_ids: list[str] = []
    pagination_token = None
    LIMIT = 10_000

    print(f"Searching for records with {METADATA_FIELD}='{source_name}' in namespace '{NAMESPACE}'...")
    while True:
        kwargs = {
            "filter": {METADATA_FIELD: {"$eq": source_name}},
            "namespace": NAMESPACE,
            "limit": LIMIT,
        }
        if pagination_token:
            kwargs["pagination_token"] = pagination_token

        res = index.fetch_by_metadata(**kwargs)
        vectors = res.get("vectors", {}) or {}
        all_ids.extend(vectors.keys())

        pagination = res.get("pagination") or {}
        pagination_token = pagination.get("next")

        if not pagination_token or not vectors:
            break

    count = len(all_ids)
    print(
        f"Found {count} records in namespace '{NAMESPACE}' "
        f"with {METADATA_FIELD}='{source_name}'."
    )

    if count == 0:
        print("Nothing to delete. Done.")
    else:
        confirm = input("Delete ALL of them? (Y/N): ").strip().lower()
        if not confirm.startswith("y"):
            print("Deletion cancelled.")
        else:
            print("Deleting records...")
            BATCH = 1000
            for i in range(0, count, BATCH):
                batch_ids = all_ids[i:i + BATCH]
                index.delete(
                    ids=batch_ids,
                    namespace=NAMESPACE,
                )
                print(f"  Deleted {min(i + BATCH, count)}/{count}...", end="\r")

            print(f"\nDelete requests sent for all {count} records.")

            verify_res = index.fetch_by_metadata(
                filter={METADATA_FIELD: {"$eq": source_name}},
                namespace=NAMESPACE,
                limit=1,
            )
            remaining = len(verify_res.get("vectors", {}) or {})

            if remaining == 0:
                print(
                    f"Verification: 0 records remaining with {METADATA_FIELD}='{source_name}' "
                    f"in namespace '{NAMESPACE}'. ✅"
                )
            else:
                print(
                    f"Verification: still found {remaining} matching record(s). "
                    "This could be eventual consistency — check again shortly."
                )
